In [ ]:
from pathlib import Path
import os
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")


# Phase 5 — Algorithm Selection & Cluster Number Determination

**Project:** Wholesale Customer Segmentation

Phase 5 - Algorithm Selection & Cluster Number Determination
Wholesale Customers Clustering Analysis

Steps covered (per implementation plan):
 11. Choose algorithm(s) to compare (K-Means + Hierarchical)
 12. Test K = 2-10
 13. Elbow Method
 14. Silhouette Analysis
 15. Select the final K

Depends on: phase4_transform_scale.py output (scaled_features.csv)
Outputs: PNG figures saved to ./figures/, printed diagnostics, documented K decision

Assumptions (stated explicitly, not guessed):
 - K range: 2-10, per the implementation plan.
 - K-Means: random_state=42, n_init=20 for explicit reproducibility.
 - Agglomerative Hierarchical Clustering: Ward linkage (standard default for
   Euclidean space, roughly-equal-variance clusters) — not specified in the
   plan, so flagged here rather than silently chosen.

### How to use this notebook
Run cells from top to bottom. Keep the project files in the same folder as this notebook. Phases 1–4 create the data preparation artifacts used by later phases; Phases 5–9 read those artifacts; Phase 10 assembles the final report.

In [ ]:
from pathlib import Path
import os
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
os.chdir(PROJECT_ROOT)
print(f"Project root: {PROJECT_ROOT}")


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score, silhouette_samples
from scipy.cluster.hierarchy import dendrogram, linkage



# Only the figure directory is needed from the earlier phase.
FIG_DIR = "figures"
import os
os.makedirs(FIG_DIR, exist_ok=True)

sns.set_style("whitegrid")

RANDOM_STATE = 42
K_RANGE = range(2, 11)  # 2 to 10 inclusive
LINKAGE_METHOD = "ward"
N_INIT = 20

## SECTION 1: Load Transformed & Scaled Feature Matrix

In [ ]:
# SECTION 1: Load Transformed & Scaled Feature Matrix
# ===========================================================================
def load_scaled_features() -> pd.DataFrame:
    """Load the log-transformed, standardized feature matrix produced by
    Phase 4. This is the exact input both algorithms will cluster on."""
    print("=" * 70)
    print("SECTION 1: LOAD TRANSFORMED & SCALED FEATURE MATRIX")
    print("=" * 70)

    scaled_df = pd.read_csv("scaled_features.csv")
    print(f"Loaded scaled_features.csv — shape: {scaled_df.shape}")
    print(f"Columns: {list(scaled_df.columns)}")
    print(f"Mean per column (expect ~0): {scaled_df.mean().round(3).to_dict()}")
    print(f"Std per column (expect ~1): {scaled_df.std().round(3).to_dict()}")
    return scaled_df

## SECTION 2: K-Means — Fit Across K = 2 to 10 (Steps 11, 12)

In [ ]:
# SECTION 2: K-Means — Fit Across K = 2 to 10 (Steps 11, 12)
# ===========================================================================
def fit_kmeans_range(X: np.ndarray) -> pd.DataFrame:
    """Fit K-Means for each K in K_RANGE, recording inertia (for the elbow
    method) and average silhouette score (for silhouette analysis)."""
    print("\n" + "=" * 70)
    print("SECTION 2: K-MEANS — FIT ACROSS K = 2 TO 10")
    print("=" * 70)

    rows = []
    for k in K_RANGE:
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=N_INIT)
        labels = km.fit_predict(X)
        sil = silhouette_score(X, labels)
        rows.append({"k": k, "inertia": km.inertia_, "silhouette": sil})
        print(f"  K={k:2d}  inertia={km.inertia_:10.2f}  silhouette={sil:.4f}")

    kmeans_results = pd.DataFrame(rows)
    print("\n[OK] K-Means fitted for all K in range with fixed random_state="
          f"{RANDOM_STATE}, n_init={N_INIT} for reproducibility.")
    return kmeans_results

## SECTION 3: K-Means — Elbow Method Plot (Step 13)

In [ ]:
# SECTION 3: K-Means — Elbow Method Plot (Step 13)
# ===========================================================================
def plot_elbow(kmeans_results: pd.DataFrame) -> None:
    """Plot inertia vs. K to visually identify the 'elbow' point."""
    print("\n" + "=" * 70)
    print("SECTION 3: K-MEANS — ELBOW METHOD PLOT")
    print("=" * 70)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(kmeans_results["k"], kmeans_results["inertia"], marker="o", color="steelblue")
    ax.set_xlabel("Number of Clusters (K)")
    ax.set_ylabel("Inertia (Within-Cluster Sum of Squares)")
    ax.set_title("Elbow Method: Inertia vs. K (K-Means)")
    ax.set_xticks(list(K_RANGE))
    fig.tight_layout()
    fig.savefig(f"{FIG_DIR}/09_elbow_method.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] Saved {FIG_DIR}/09_elbow_method.png")

    # Rough automatic elbow flag: largest drop in the rate of decrease
    diffs = np.diff(kmeans_results["inertia"].values)
    diffs2 = np.diff(diffs)  # second derivative — elbow = point of max curvature
    if len(diffs2) > 0:
        elbow_idx = np.argmax(diffs2) + 2  # +2 offset for double diff on K starting at 2
        elbow_k = kmeans_results["k"].iloc[elbow_idx] if elbow_idx < len(kmeans_results) else None
        print(f"[NOTE] Approximate elbow (max-curvature heuristic) around K={elbow_k}. "
              "Treat this as a rough guide only; the elbow is gradual here, so silhouette, "
              "cross-algorithm comparison, and stability are used for the final decision.")

## SECTION 4: Hierarchical Clustering — Fit Across K = 2 to 10 (Steps 11, 12)

In [ ]:
# SECTION 4: Hierarchical Clustering — Fit Across K = 2 to 10 (Steps 11, 12)
# ===========================================================================
def fit_hierarchical_range(X: np.ndarray) -> pd.DataFrame:
    """Fit Agglomerative Hierarchical Clustering (Ward linkage) for each K
    in K_RANGE, recording average silhouette score."""
    print("\n" + "=" * 70)
    print("SECTION 4: HIERARCHICAL CLUSTERING — FIT ACROSS K = 2 TO 10")
    print("=" * 70)
    print(f"Linkage method: {LINKAGE_METHOD} (assumption — not specified in plan)")

    rows = []
    for k in K_RANGE:
        agg = AgglomerativeClustering(n_clusters=k, linkage=LINKAGE_METHOD)
        labels = agg.fit_predict(X)
        sil = silhouette_score(X, labels)
        rows.append({"k": k, "silhouette": sil})
        print(f"  K={k:2d}  silhouette={sil:.4f}")

    hier_results = pd.DataFrame(rows)
    print("\n[NOTE] Hierarchical clustering has no 'inertia' concept, so the "
          "elbow method (Step 13) applies to K-Means only, as specified in "
          "the plan. Silhouette score is the comparison metric for both.")
    return hier_results

## SECTION 5: Hierarchical Clustering — Dendrogram (supports Step 11 rationale)

In [ ]:
# SECTION 5: Hierarchical Clustering — Dendrogram (supports Step 11 rationale)
# ===========================================================================
def plot_dendrogram(X: np.ndarray) -> None:
    """Plot the Ward-linkage dendrogram to visually cross-check how many
    natural groupings the hierarchy suggests, independent of a chosen K."""
    print("\n" + "=" * 70)
    print("SECTION 5: HIERARCHICAL CLUSTERING — DENDROGRAM")
    print("=" * 70)

    Z = linkage(X, method=LINKAGE_METHOD)

    fig, ax = plt.subplots(figsize=(14, 6))
    dendrogram(Z, ax=ax, truncate_mode="lastp", p=30,
               leaf_rotation=90, leaf_font_size=8, show_contracted=True)
    ax.set_title(f"Dendrogram ({LINKAGE_METHOD.title()} Linkage, last 30 merges shown)")
    ax.set_xlabel("Customer clusters (or individual customers)")
    ax.set_ylabel("Distance (Ward)")
    fig.tight_layout()
    fig.savefig(f"{FIG_DIR}/10_dendrogram.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] Saved {FIG_DIR}/10_dendrogram.png")
    print("[NOTE] Dendrogram is a visual cross-check on cluster structure, not "
          "itself a numeric criterion for choosing K — used alongside silhouette.")

## SECTION 6: Silhouette Analysis — Comparison Plot (Step 14)

In [ ]:
# SECTION 6: Silhouette Analysis — Comparison Plot (Step 14)
# ===========================================================================
def plot_silhouette_comparison(kmeans_results: pd.DataFrame, hier_results: pd.DataFrame) -> None:
    """Plot average silhouette score vs. K for both algorithms on one chart —
    the primary decision-making figure per the plan ('more decisive than
    elbow given the skewed data')."""
    print("\n" + "=" * 70)
    print("SECTION 6: SILHOUETTE ANALYSIS — COMPARISON PLOT")
    print("=" * 70)

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(kmeans_results["k"], kmeans_results["silhouette"], marker="o",
            label="K-Means", color="steelblue")
    ax.plot(hier_results["k"], hier_results["silhouette"], marker="s",
            label="Hierarchical (Ward)", color="darkorange")
    ax.set_xlabel("Number of Clusters (K)")
    ax.set_ylabel("Average Silhouette Score")
    ax.set_title("Silhouette Score vs. K: K-Means vs. Hierarchical Clustering")
    ax.set_xticks(list(K_RANGE))
    ax.legend()
    fig.tight_layout()
    fig.savefig(f"{FIG_DIR}/11_silhouette_comparison.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] Saved {FIG_DIR}/11_silhouette_comparison.png")

    combined = kmeans_results[["k", "silhouette"]].merge(
        hier_results[["k", "silhouette"]], on="k", suffixes=("_kmeans", "_hierarchical"))
    print("\nSilhouette scores side by side:")
    print(combined.round(4).to_string(index=False))

## SECTION 7: Detailed Silhouette Plot for Top Candidate K Values (Step 14)

In [ ]:
# SECTION 7: Detailed Silhouette Plot for Top Candidate K Values (Step 14)
# ===========================================================================
def plot_detailed_silhouette(X: np.ndarray, k: int) -> float:
    """Per-sample silhouette plot for a specific K (K-Means), showing cluster
    sizes and whether any cluster's samples fall below the average score —
    a more rigorous silhouette diagnostic than the average alone."""
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=N_INIT)
    labels = km.fit_predict(X)
    sil_avg = silhouette_score(X, labels)
    sample_sil = silhouette_samples(X, labels)

    fig, ax = plt.subplots(figsize=(8, 6))
    y_lower = 10
    for i in range(k):
        cluster_sil = np.sort(sample_sil[labels == i])
        size = cluster_sil.shape[0]
        y_upper = y_lower + size
        color = cm.nipy_spectral(float(i) / k)
        ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_sil,
                          facecolor=color, edgecolor=color, alpha=0.7)
        ax.text(-0.05, y_lower + 0.5 * size, str(i))
        y_lower = y_upper + 10

    ax.axvline(x=sil_avg, color="red", linestyle="--", label=f"Average = {sil_avg:.3f}")
    ax.set_title(f"Detailed Silhouette Plot — K-Means, K={k}")
    ax.set_xlabel("Silhouette Coefficient")
    ax.set_ylabel("Cluster")
    ax.legend()
    fig.tight_layout()
    fig.savefig(f"{FIG_DIR}/12_detailed_silhouette_k{k}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[OK] Saved {FIG_DIR}/12_detailed_silhouette_k{k}.png (avg silhouette={sil_avg:.4f})")
    return sil_avg


def run_detailed_silhouette_for_top_candidates(X: np.ndarray, kmeans_results: pd.DataFrame) -> None:
    """Generate detailed silhouette plots for the top 2 K-Means candidates
    by average silhouette score, to support the final K decision."""
    print("\n" + "=" * 70)
    print("SECTION 7: DETAILED SILHOUETTE PLOTS FOR TOP CANDIDATE K VALUES")
    print("=" * 70)

    top_candidates = kmeans_results.sort_values("silhouette", ascending=False).head(2)["k"].tolist()
    print(f"Top 2 K-Means candidates by average silhouette score: {top_candidates}")
    for k in top_candidates:
        plot_detailed_silhouette(X, k)

## SECTION 8: Final K Selection & Documented Reasoning (Step 15)

In [ ]:
# SECTION 8: Final K Selection & Documented Reasoning (Step 15)
# ===========================================================================
def select_final_k(kmeans_results: pd.DataFrame, hier_results: pd.DataFrame) -> dict:
    """Combine elbow + silhouette diagnostics from both algorithms to select
    and document the final K, per the plan's requirement to avoid picking K
    solely because it is a 'nice' number."""
    print("\n" + "=" * 70)
    print("SECTION 8: FINAL K SELECTION & DOCUMENTED REASONING")
    print("=" * 70)

    best_kmeans_row = kmeans_results.loc[kmeans_results["silhouette"].idxmax()]
    best_hier_row = hier_results.loc[hier_results["silhouette"].idxmax()]

    best_k_kmeans = int(best_kmeans_row["k"])
    best_k_hier = int(best_hier_row["k"])

    print(f"Best K by K-Means silhouette: K={best_k_kmeans} (silhouette={best_kmeans_row['silhouette']:.4f})")
    print(f"Best K by Hierarchical silhouette: K={best_k_hier} (silhouette={best_hier_row['silhouette']:.4f})")

    agreement = (best_k_kmeans == best_k_hier)
    final_k = best_k_kmeans  # K-Means is the primary algorithm per the plan

    # Trade-off check: a very low K can be statistically attractive but may be too coarse to satisfy the task's requirement for an "in-depth"
    # multi-cluster business analysis. This is reported honestly rather than
    # silently overridden in either direction.
    coarse_k_warning = ""
    if final_k <= 3:
        runner_up = kmeans_results[kmeans_results["k"] > final_k].sort_values(
            "silhouette", ascending=False).iloc[0]
        coarse_k_warning = (
            f"\nIMPORTANT TRADE-OFF TO FLAG: K={final_k} is the statistically optimal "
            "split by silhouette score, but such a low K risks producing clusters "
            "too broad for a business-actionable, in-depth cluster analysis (it may "
            "largely mirror the existing binary Channel variable rather than reveal "
            "new structure). The next-best K by silhouette in the tested range is "
            f"K={int(runner_up['k'])} (silhouette={runner_up['silhouette']:.4f}), only "
            f"{best_kmeans_row['silhouette'] - runner_up['silhouette']:.4f} lower. "
            "This trade-off between statistical separation and business granularity is reported transparently. "
            "Phase 6 stability testing is used as an additional criterion before the primary solution is finalized.\n"
        )
        print(coarse_k_warning)

    reasoning = (
        f"FINAL K SELECTED: {final_k}\n\n"
        "REASONING:\n"
        f"  1. K-Means silhouette score peaks at K={best_k_kmeans} "
        f"({best_kmeans_row['silhouette']:.4f}), the highest average silhouette "
        "across the full K=2-10 range tested — silhouette is prioritized over "
        "the elbow plot per the plan, since the elbow is harder to read "
        "precisely on this skewed, transformed data.\n"
        f"  2. Hierarchical clustering (Ward linkage) provides an independent cross-check across K values; "
        f"its best silhouette occurs at K={best_k_hier} ({best_hier_row['silhouette']:.4f}), "
        + ("which agrees with the K-Means result. This agreement strengthens confidence in the selected candidate, "
           "but is not proof that K=2 is the unique true number of clusters.\n"
           if agreement else
           "which DIFFERS from the K-Means result. K-Means is retained as the "
           "primary algorithm (per the plan) since it is the deliverable's "
           "stated main method; the hierarchical result is reported as a "
           "secondary cross-check and discussed as a limitation/alternative "
           "view in the final report rather than silently discarded.\n")
        + f"  3. K={final_k} was NOT chosen because it is a conventionally "
        "'nice' number (e.g. K=3 to match Channel/Region cardinality) — it "
        "is chosen strictly because it maximizes the measured silhouette "
        "score, which is the plan's specified decisive criterion.\n"
        "  4. The elbow plot (Section 3) is reported alongside as a "
        "supporting/sanity-check visualization, consistent with the plan "
        "treating it as informative but non-decisive versus silhouette.\n"
    )
    print(reasoning)

    return {
        "final_k": final_k,
        "best_k_kmeans": best_k_kmeans,
        "best_k_hierarchical": best_k_hier,
        "kmeans_silhouette_at_final_k": float(best_kmeans_row["silhouette"]),
        "algorithms_agree": agreement,
        "coarse_k_flagged": bool(coarse_k_warning),
    }

## MAIN — run Phase 5 end to end

In [ ]:
# MAIN — run Phase 5 end to end
# ===========================================================================
if __name__ == "__main__":
    scaled_df = load_scaled_features()
    X = scaled_df.values

    kmeans_results = fit_kmeans_range(X)
    plot_elbow(kmeans_results)

    hier_results = fit_hierarchical_range(X)
    plot_dendrogram(X)

    plot_silhouette_comparison(kmeans_results, hier_results)
    run_detailed_silhouette_for_top_candidates(X, kmeans_results)

    k_decision = select_final_k(kmeans_results, hier_results)

    print("\n" + "=" * 70)
    print("PHASE 5 COMPLETE")
    print("=" * 70)
    print(f"[OK] K-Means and Hierarchical Clustering tested for K=2-10.")
    print(f"[OK] Elbow method, silhouette analysis, hierarchical cross-check, and detailed silhouette diagnostics computed.")
    print(f"[OK] Final K selected: {k_decision['final_k']} "
          f"(K-Means silhouette={k_decision['kmeans_silhouette_at_final_k']:.4f}, "
          f"algorithms agree: {k_decision['algorithms_agree']}).")
    print("[OK] Ready for Phase 6 (Final Model Training & Validation).")

### Phase 5 checkpoint

Review the outputs and figures generated by this phase before moving to the next phase.